In [74]:
# coding: utf-8
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', lambda x: '%.2f' % x)


## 共享单车用户行为分析
### Step1: 数据加载与初步探索

#### 1. 加载共享单车数据

In [75]:
df = pd.read_csv('../data/上海共享单车数据.csv')
df.head()

,orderid,bikeid,userid,start_time,start_location_x,start_location_y,end_time,end_location_x,end_location_y,track
0,78387,158357,10080,2016-08-20 06:57,121.35,31.39,2016-08-20 07:04,121.36,31.39,"121.347,31.392#121.348,31.389#121.349,31.390#1..."
1,891333,92776,6605,2016-08-29 19:09,121.51,31.28,2016-08-29 19:31,121.49,31.27,"121.489,31.270#121.489,31.271#121.490,31.270#1..."
2,1106623,152045,8876,2016-08-13 16:17,121.38,31.25,2016-08-13 16:36,121.41,31.25,"121.381,31.251#121.382,31.251#121.382,31.252#1..."
3,1389484,196259,10648,2016-08-23 21:34,121.48,31.32,2016-08-23 21:43,121.47,31.32,"121.471,31.325#121.472,31.325#121.473,31.324#1..."
4,188537,78208,11735,2016-08-16 07:32,121.41,31.29,2016-08-16 07:41,121.42,31.29,"121.407,31.291#121.407,31.292#121.408,31.291#1..."


In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102361 entries, 0 to 102360
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   orderid           102361 non-null  int64  
 1   bikeid            102361 non-null  int64  
 2   userid            102361 non-null  int64  
 3   start_time        102361 non-null  object 
 4   start_location_x  102361 non-null  float64
 5   start_location_y  102361 non-null  float64
 6   end_time          102361 non-null  object 
 7   end_location_x    102361 non-null  float64
 8   end_location_y    102361 non-null  float64
 9   track             102361 non-null  object 
dtypes: float64(4), int64(3), object(3)
memory usage: 7.8+ MB


In [77]:
df.describe()

,orderid,bikeid,userid,start_location_x,start_location_y,end_location_x,end_location_y
count,102361.00,102361.00,102361.00,102361.00,102361.00,102361.00,102361.00
mean,894083.80,155425.87,7834.44,121.45,31.25,121.45,31.25
std,521965.12,98816.63,4530.39,0.06,0.06,0.06,0.06
min,6.00,3.00,1.00,121.17,30.84,120.49,30.84
25%,438679.00,74701.00,4095.00,121.42,31.21,121.41,31.21
50%,890151.00,141990.00,7617.00,121.46,31.26,121.46,31.26
75%,1343252.00,228303.00,11278.00,121.50,31.29,121.50,31.29
max,1807864.00,387351.00,17753.00,121.97,31.45,121.97,31.48


#### 2. 加载上海天气数据

In [78]:
df_weather = pd.read_excel('../data/上海天气.xlsx')
df_weather.head()

,日期,最高温,最低温,天气,风力风向,日照时长
0,2016-08-01 周一,31.8℃,27.2℃,晴,东南风3级,12.0小时
1,2016-08-02 周二,32.3℃,27.9℃,毛毛雨,东南风3级,10.5小时
2,2016-08-03 周三,28.7℃,26.1℃,中雨,西南风3级,0.4小时
3,2016-08-04 周四,31.5℃,25.2℃,中雨,北风2级,10.0小时
4,2016-08-05 周五,30.9℃,26℃,毛毛雨,北风2级,7.5小时


In [79]:
df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   日期      31 non-null     object
 1   最高温     31 non-null     object
 2   最低温     31 non-null     object
 3   天气      31 non-null     object
 4   风力风向    31 non-null     object
 5   日照时长    31 non-null     object
dtypes: object(6)
memory usage: 1.6+ KB


### Step2: 数据清洗

#### 1. 处理缺失值

In [80]:
# 查看共享单车数据缺失值
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    '字段': missing_count.index,
    '缺失数量': missing_count.values,
    '缺失率': missing_pct.values
})
missing_df[missing_df['缺失数量'] > 0]

,字段,缺失数量,缺失率


In [81]:
# 天气缺失值
dfw = df_weather.copy()
dfw.isnull().sum()

日期      0
最高温     0
最低温     0
天气      0
风力风向    0
日照时长    0
dtype: int64

#### 2. 数据类型转换

In [82]:
# 时间字段转换为datetime类型
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

df['start_hour'] = df['start_time'].dt.hour
df['start_weekday'] = df['start_time'].dt.weekday
df['start_day'] = df['start_time'].dt.day
df['duration_min'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60

df[['start_time','end_time','start_hour','start_weekday','duration_min']].head()

,start_time,end_time,start_hour,start_weekday,duration_min
0,2016-08-20 06:57:00,2016-08-20 07:04:00,6,5,7.00
1,2016-08-29 19:09:00,2016-08-29 19:31:00,19,0,22.00
2,2016-08-13 16:17:00,2016-08-13 16:36:00,16,5,19.00
3,2016-08-23 21:34:00,2016-08-23 21:43:00,21,1,9.00
4,2016-08-16 07:32:00,2016-08-16 07:41:00,7,1,9.00


#### 3. 异常值处理

In [83]:
df['duration_min'].describe()

count   102361.00
mean        17.20
std         34.05
min          1.00
25%          7.00
50%         12.00
75%         20.00
max       4725.00
Name: duration_min, dtype: float64

In [84]:
# 处理异常骑行时长：<1分钟或>180分钟视为异常
print(f'原始数据量: {len(df)}')
exp1 = len(df[df['duration_min'] < 1])
exp2 = len(df[df['duration_min'] > 180])
print(f'时长<1分钟: {exp1}')
print(f'时长>180分钟: {exp2}')
df = df[(df['duration_min'] >= 1) & (df['duration_min'] <= 180)]
print(f'清洗后数据量: {len(df)}')

原始数据量: 102361
时长<1分钟: 0
时长>180分钟: 148
清洗后数据量: 102213


In [85]:
# 骑行距离计算及异常值处理（距离>72km(180min*400m/min)视为异常）
# 上海纬度~31.2°, 1°经度≈95km, 1°纬度=111km
df['distance_km'] = np.sqrt(
    ((df['end_location_x'] - df['start_location_x']) * 95)**2 +
    ((df['end_location_y'] - df['start_location_y']) * 111)**2
).round(3)
df['speed_kmh'] = (df['distance_km'] / (df['duration_min'] / 60)).round(2)

print('运行检查结果:')
print(f'起始经度: {df["start_location_x"].min():.3f} ~ {df["start_location_x"].max():.3f}')
print(f'起始纬度: {df["start_location_y"].min():.3f} ~ {df["start_location_y"].max():.3f}')
print(f'结束经度: {df["end_location_x"].min():.3f} ~ {df["end_location_x"].max():.3f}')
print(f'结束纬度: {df["end_location_y"].min():.3f} ~ {df["end_location_y"].max():.3f}')
print(f'距离范围: {df["distance_km"].min():.3f} ~ {df["distance_km"].max():.3f} km')
print(f'速度范围: {df["speed_kmh"].min():.2f} ~ {df["speed_kmh"].max():.2f} km/h')

outlier_dist = len(df[df['distance_km'] > 72])
print(f'\n距离>72km的异常数据: {outlier_dist} 条')
df = df[df['distance_km'] <= 72]
print(f'清洗后数据量: {len(df)}')

运行检查结果:
起始经度: 121.173 ~ 121.970
起始纬度: 30.842 ~ 31.450
结束经度: 120.486 ~ 121.971
结束纬度: 30.841 ~ 31.477
距离范围: 0.146 ~ 93.244 km
速度范围: 0.05 ~ 472.40 km/h

距离>72km的异常数据: 3 条
清洗后数据量: 102210


#### 4. 重复值处理

In [86]:
# 检查重复订单
print(f'orderid重复数: {df.duplicated(subset="orderid").sum()}')
df = df.drop_duplicates(subset='orderid')
print(f'去重后数据量: {len(df)}')

orderid重复数: 0
去重后数据量: 102210


### Step3: 特征工程

#### 1. 骑行特征构造

In [87]:
# 时间段分类函数
def get_time_period(hour):
    if 6 <= hour < 9:
        return '早高峰(6-9点)'
    elif 17 <= hour < 20:
        return '晚高峰(17-20点)'
    elif 9 <= hour < 17:
        return '日间(9-17点)'
    else:
        return '夜间(20-6点)'
df['time_period'] = df['start_hour'].apply(get_time_period)
df['is_weekend'] = df['start_weekday'].apply(lambda x: 1 if x >= 5 else 0)
df[['time_period','is_weekend']].head()

,time_period,is_weekend
0,早高峰(6-9点),1
1,晚高峰(17-20点),0
2,日间(9-17点),1
3,夜间(20-6点),0
4,早高峰(6-9点),0


#### 2. 天气数据清洗

In [88]:
# 处理天气数据
dfw = df_weather.copy()
dfw['date'] = pd.to_datetime(dfw['日期'].str.extract(r'(\d{4}-\d{2}-\d{2})', expand=False))
dfw['max_temp'] = dfw['最高温'].str.replace('℃', '').astype(float)
dfw['min_temp'] = dfw['最低温'].str.replace('℃', '').astype(float)
dfw['avg_temp'] = ((dfw['max_temp'] + dfw['min_temp']) / 2).round(1)
dfw['sunshine_h'] = dfw['日照时长'].str.replace('小时', '').astype(float)
dfw[['date','avg_temp','天气','sunshine_h']].head()

,date,avg_temp,天气,sunshine_h
0,2016-08-01,29.50,晴,12.00
1,2016-08-02,30.10,毛毛雨,10.50
2,2016-08-03,27.40,中雨,0.40
3,2016-08-04,28.40,中雨,10.00
4,2016-08-05,28.40,毛毛雨,7.50


#### 3. 合并天气与骑行数据

In [89]:
# 合并天气和骑行数据
df['ride_date'] = df['start_time'].dt.date
dfw['date_only'] = dfw['date'].dt.date
df = df.merge(dfw[['date_only','天气','avg_temp','sunshine_h','max_temp','min_temp']],
              left_on='ride_date', right_on='date_only', how='left')
df = df.drop(columns=['date_only'])
print(f'合并后数据量: {len(df)}')
print(f'列数: {len(df.columns)}')
df[['start_time','天气','avg_temp','sunshine_h']].head()

合并后数据量: 102210
列数: 24


,start_time,天气,avg_temp,sunshine_h
0,2016-08-20 06:57:00,晴,31.00,11.90
1,2016-08-29 19:09:00,晴,25.20,12.00
2,2016-08-13 16:17:00,晴,29.60,12.00
3,2016-08-23 21:34:00,晴,28.90,11.90
4,2016-08-16 07:32:00,晴,29.20,12.00


In [90]:
# 按时间段取更真实的温度值
# 根据start_hour决定取哪种温度：
#  20-7点(夜间)  → 最低温
#   7-11点(上午)  → 平均温
#  11-17点(下午)  → 最高温
#  17-20点(傍晚)  → 平均温
def get_actual_temp(row):
    h = row['start_hour']
    if (h >= 20) or (h < 7):
        return row['min_temp']       # 夜间取最低
    elif 7 <= h < 11:
        return row['avg_temp']       # 上午取平均
    elif 11 <= h < 17:
        return row['max_temp']       # 下午取最高
    else:  # 17 <= h < 20
        return row['avg_temp']       # 傍晚取平均

df['actual_temp'] = df.apply(get_actual_temp, axis=1).round(1)

# 展示各时段取值对比
df[['start_time', 'start_hour', 'min_temp', 'max_temp', 'avg_temp', 'actual_temp']].head(20)


,start_time,start_hour,min_temp,max_temp,avg_temp,actual_temp
0,2016-08-20 06:57:00,6,28.10,33.80,31.00,28.10
1,2016-08-29 19:09:00,19,21.00,29.50,25.20,25.20
2,2016-08-13 16:17:00,16,27.40,31.90,29.60,31.90
3,2016-08-23 21:34:00,21,26.20,31.60,28.90,26.20
4,2016-08-16 07:32:00,7,26.60,31.80,29.20,29.20
5,2016-08-07 21:00:00,21,26.50,31.20,28.80,26.50
6,2016-08-29 13:39:00,13,21.00,29.50,25.20,29.50
7,2016-08-31 08:16:00,8,20.00,31.50,25.80,25.80
8,2016-08-30 12:49:00,12,20.50,29.40,25.00,29.40
9,2016-08-09 19:51:00,19,27.00,30.40,28.70,28.70


#### 4. 用户行为特征

In [91]:
# 按用户聚合
pre_time = pd.to_datetime('2016-09-01')
user_features = df.groupby('userid').agg({
    'start_time': lambda x: (pre_time - x.max()).days,
    'orderid': 'count',
    'distance_km': 'sum',
    'duration_min': ['sum', 'mean'],
    'speed_kmh': 'mean'
}).round(2)
user_features.columns = ['recency','frequency','total_distance',
                         'total_duration','avg_duration','avg_speed']
user_features = user_features.reset_index()
user_features.head()

,userid,recency,frequency,total_distance,total_duration,avg_duration,avg_speed
0,1,0,5,7.18,87.00,17.40,5.87
1,3,3,8,8.49,108.00,13.50,4.81
2,6,0,2,2.58,29.00,14.50,5.68
3,7,6,8,10.49,118.00,14.75,5.37
4,8,4,7,7.09,82.00,11.71,6.01


In [92]:
user_features.describe()

,userid,recency,frequency,total_distance,total_duration,avg_duration,avg_speed
count,16883.00,16883.00,16883.00,16883.00,16883.00,16883.00,16883.00
mean,8734.93,4.14,6.05,9.43,99.85,16.65,6.57
std,5039.44,5.27,3.48,6.43,69.31,8.72,1.37
min,1.00,0.00,1.00,0.15,2.00,2.00,0.09
25%,4388.50,0.00,3.00,4.52,48.00,11.50,5.83
50%,8723.00,2.00,6.00,8.23,86.00,15.00,6.60
75%,13033.50,6.00,8.00,12.96,138.00,19.57,7.34
max,17753.00,30.00,25.00,52.77,511.00,153.00,17.74


#### 5. 单车使用特征

In [93]:
# 按单车聚合
bike_features = df.groupby('bikeid').agg({
    'orderid': 'count',
    'distance_km': ['sum', 'mean'],
    'duration_min': ['sum', 'mean'],
}).round(2)
bike_features.columns = ['ride_count','total_distance','avg_distance',
                         'total_duration','avg_duration']
bike_features = bike_features.reset_index()
bike_features.head()

,bikeid,ride_count,total_distance,avg_distance,total_duration,avg_duration
0,3,1,2.14,2.14,20.00,20.00
1,6,1,0.38,0.38,4.00,4.00
2,13,2,0.88,0.44,7.00,3.50
3,22,1,0.87,0.87,12.00,12.00
4,25,1,2.01,2.01,21.00,21.00


In [94]:
# 单车使用分布
print(bike_features['ride_count'].describe())
print(f'\n使用1次的单车: {len(bike_features[bike_features["ride_count"]==1])} 辆')
print(f'使用5次以上的单车: {len(bike_features[bike_features["ride_count"]>=5])} 辆')

count   78985.00
mean        1.29
std         0.67
min         1.00
25%         1.00
50%         1.00
75%         1.00
max        11.00
Name: ride_count, dtype: float64

使用1次的单车: 62121 辆
使用5次以上的单车: 439 辆


#### 6. 保存处理后的数据

In [ ]:
# 字段名标注单位
rename_map = {
    'max_temp': 'max_temp(℃)',
    'min_temp': 'min_temp(℃)',
    'avg_temp': 'avg_temp(℃)',
    'actual_temp': 'actual_temp(℃)',
    'start_location_x': 'start_location_x(°)',
    'start_location_y': 'start_location_y(°)',
    'end_location_x': 'end_location_x(°)',
    'end_location_y': 'end_location_y(°)',
    'sunshine_h': 'sunshine_h(h)',
    'start_hour': 'start_hour(h)',
}
df = df.rename(columns=rename_map)

# 用户特征加单位
user_rename = {
    'recency': 'recency(day)',
    'frequency': 'frequency(次)',
    'total_distance': 'total_distance(km)',
    'total_duration': 'total_duration(min)',
    'avg_duration': 'avg_duration(min)',
    'avg_speed': 'avg_speed(km/h)',
}
user_features = user_features.rename(columns=user_rename)

# 单车特征加单位
bike_rename = {
    'ride_count': 'ride_count(次)',
    'total_distance': 'total_distance(km)',
    'avg_distance': 'avg_distance(km)',
    'total_duration': 'total_duration(min)',
    'avg_duration': 'avg_duration(min)',
}
bike_features = bike_features.rename(columns=bike_rename)

# 天气数据加单位
dfw = dfw.rename(columns={
    'max_temp': 'max_temp(℃)',
    'min_temp': 'min_temp(℃)',
    'avg_temp': 'avg_temp(℃)',
    'sunshine_h': 'sunshine_h(h)',
})

print('字段名已更新为带单位格式:')
print([c for c in df.columns if '(' in c])


In [95]:
# 保存清洗和特征工程后的数据
df.to_csv('../data/共享单车_清洗后.csv', index=False, encoding='utf-8-sig')
user_features.to_csv('../data/共享单车_用户特征.csv', index=False, encoding='utf-8-sig')
bike_features.to_csv('../data/共享单车_单车特征.csv', index=False, encoding='utf-8-sig')
dfw.to_csv('../data/共享单车_天气清洗后.csv', index=False, encoding='utf-8-sig')
print('数据保存完成！')
print(f'骑行订单数据: {df.shape}')
print(f'用户特征数据: {user_features.shape}')
print(f'单车特征数据: {bike_features.shape}')

数据保存完成！
骑行订单数据: (102210, 25)
用户特征数据: (16883, 7)
单车特征数据: (78985, 6)


In [96]:
df.head()

,orderid,bikeid,userid,start_time,start_location_x,start_location_y,end_time,end_location_x,end_location_y,track,...,speed_kmh,time_period,is_weekend,ride_date,天气,avg_temp,sunshine_h,max_temp,min_temp,actual_temp
0,78387,158357,10080,2016-08-20 06:57:00,121.35,31.39,2016-08-20 07:04:00,121.36,31.39,"121.347,31.392#121.348,31.389#121.349,31.390#1...",...,7.39,早高峰(6-9点),1,2016-08-20,晴,31.00,11.90,33.80,28.10,28.10
1,891333,92776,6605,2016-08-29 19:09:00,121.51,31.28,2016-08-29 19:31:00,121.49,31.27,"121.489,31.270#121.489,31.271#121.490,31.270#1...",...,5.49,晚高峰(17-20点),0,2016-08-29,晴,25.20,12.00,29.50,21.00,25.20
2,1106623,152045,8876,2016-08-13 16:17:00,121.38,31.25,2016-08-13 16:36:00,121.41,31.25,"121.381,31.251#121.382,31.251#121.382,31.252#1...",...,6.93,日间(9-17点),1,2016-08-13,晴,29.60,12.00,31.90,27.40,31.90
3,1389484,196259,10648,2016-08-23 21:34:00,121.48,31.32,2016-08-23 21:43:00,121.47,31.32,"121.471,31.325#121.472,31.325#121.473,31.324#1...",...,9.03,夜间(20-6点),0,2016-08-23,晴,28.90,11.90,31.60,26.20,26.20
4,188537,78208,11735,2016-08-16 07:32:00,121.41,31.29,2016-08-16 07:41:00,121.42,31.29,"121.407,31.291#121.407,31.292#121.408,31.291#1...",...,7.57,早高峰(6-9点),0,2016-08-16,晴,29.20,12.00,31.80,26.60,29.20
